# Experiment 5: PKNORLEN OFN under test-time perturbation

Evaluate this stock's fixed experiment 5 model on its original October-December 2017 test days at **P = 1%, 5%, 10%, 25%, and 50%**. Reuse its saved clean P=0% scores. Keep labels, 50-snapshot windows, k=20, and batch size 64 fixed; do not retrain.

Run this notebook independently in the `deep-learning` environment, from the project root or this experiment directory. It needs the raw HDF, processed daily files, prepared OFN features, labels, and this stock's completed training outputs. Run one notebook at a time on the laptop. Sections 5-6 display the same overall and per-class tables as experiment 4; Section 7 saves them under `PKNORLEN/perturbation/`.


## 1. Set the stock, probabilities, and device

Use the same perturbation probabilities, seed 42, tick size 0.01, and unit added volume as experiment 4. Choose CUDA when available, otherwise the Apple GPU or CPU. Splitting by stock allows independent runs; it does not reduce the total number of test windows.

In [1]:
from datetime import time
from pathlib import Path
import sys
from time import perf_counter

import numpy as np
import pandas as pd
import torch
from IPython.display import display

PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == "experiment_05" and PROJECT_DIR.parent.name == "experiments":
    PROJECT_DIR = PROJECT_DIR.parent.parent
sys.path.insert(0, str(PROJECT_DIR))

from deeplob import DeepLOB
from prepare_data import select_trading_days
from train_deeplob import DailyWindowDataset, classification_metrics, run_epoch
from wse_lob import (
    build_daily_orderbooks,
    compute_history_statistics,
    normalize_features,
    orderbook_to_ofn,
    perturb_orderbook,
)

torch.set_num_threads(1)
STOCK = "PKNORLEN"
PERTURBATION_PROBABILITIES = (0.01, 0.05, 0.10, 0.25, 0.50)
PERTURBATION_SEED = 42
TICK_SIZE = 0.01
HISTORY_DAYS = 5
PRICE_RANGE = 0.10
MIN_LEVELS = 10
MAX_COST = 0.035
COSTS = ((11.0 ** (np.arange(1, 11) / 10) - 1) / 10) * MAX_COST
DAILY_START = time(9)
DAILY_END = time(16, 50)

EXPERIMENT_DIR = PROJECT_DIR / "experiments" / "experiment_05"
RUN_DIR = EXPERIMENT_DIR / STOCK
OUTPUT_DIR = RUN_DIR / "perturbation"
SOURCE_FILE = PROJECT_DIR / "data" / "wselob_2017_source" / "orders" / f"{STOCK}_lob_2017_zlib.h5"
PROCESSED_DIR = PROJECT_DIR / "data" / "wselob_2017_processed" / STOCK

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print(f"Python: {sys.executable}; device: {DEVICE}; stock: {STOCK}")

Python: /opt/anaconda3/envs/deep-learning/bin/python; device: mps; stock: PKNORLEN


## 2. Load the selected model and clean baseline

Use the selected checkpoint and all 62 test dates. Reconstruct the preceding five trading days to obtain clean OFN normalization statistics. The processed files provide reference timestamps and mid-prices, not the full order book.

In [2]:
checkpoint = torch.load(RUN_DIR / "best_model.pt", map_location="cpu", weights_only=True)
config = checkpoint["config"]
assert config["stock"] == STOCK and config["feature_type"] == "ofn"
assert (config["window_length"], config["n_levels"], config["horizon"], config["batch_size"]) == (50, 10, 20, 64)
with np.load(RUN_DIR / "test_metrics.npz", allow_pickle=False) as arrays:
    baseline_metrics = dict(arrays)

FEATURE_DIR = PROJECT_DIR / config["feature_dir"]
LABEL_DIR = PROJECT_DIR / config["label_dir"]
test_dates = config["data_dates"]["test"]
all_dates, day_keys, selected_days, _ = select_trading_days(
    "ofn", SOURCE_FILE, PROCESSED_DIR, test_dates[0], test_dates[-1], HISTORY_DAYS,
)
assert [all_dates[index] for index in selected_days] == test_dates
first_day = selected_days[0] - HISTORY_DAYS
for index in range(first_day, selected_days[-1] + 1):
    date = all_dates[index]
    if not (PROCESSED_DIR / f"{date}.npz").is_file():
        raise FileNotFoundError(PROCESSED_DIR / f"{date}.npz")
for date in test_dates:
    for folder in (FEATURE_DIR, LABEL_DIR):
        if not (folder / f"{date}.npz").is_file():
            raise FileNotFoundError(folder / f"{date}.npz")

model = DeepLOB(
    n_levels=config["n_levels"], batch_norm=config["batch_norm"],
    dropout=config["dropout"], initialization=config["initialization"],
)
model.load_state_dict(checkpoint["model_state_dict"])
model.requires_grad_(False)
model.to(DEVICE)
model.eval()
if DEVICE.type == "cuda":
    torch.backends.cudnn.allow_tf32 = config["cudnn_tf32"]
    torch.backends.cuda.matmul.allow_tf32 = config["matmul_tf32"]
    torch.backends.cudnn.benchmark = config["cudnn_benchmark"]
criterion = torch.nn.CrossEntropyLoss()
print(f"Loaded epoch {checkpoint['epoch']}; {len(test_dates)} test days and {HISTORY_DAYS} preceding days.")

Loaded epoch 23; 62 test days and 5 preceding days.


## 3. Reconstruct each day and perturb its snapshots

For each original snapshot, process asks before bids. Find vacant ticks between each side's original first and tenth prices, visit them in ascending order, and add volume 1 with probability P. Each probability has its own seed-42 generator that continues across test days, matching experiment 4's draws.

Keep all original deeper orders within mid-price +/-10% when calculating OFN. Add orders only to a separate snapshot map; the reconstruction state, best quotes, mid-price, and labels remain unchanged. Compute the clean features and all five perturbed versions during one replay of the day's messages. Each snapshot is perturbed once before forming overlapping windows.

In [3]:
def prepare_ofn_day(messages, expected_times, expected_mid_prices, generators):
    """Reconstruct one day once and return clean and perturbed raw OFN arrays."""
    for column in ("time", "priority_date", "order_date"):
        messages[column] = pd.to_datetime(messages[column], unit="ns", utc=True).dt.tz_convert("Europe/Warsaw")
    messages["price"] = messages["price"] / (10 ** messages["price_level"])
    orderbooks, _, _ = build_daily_orderbooks(
        messages, DAILY_START, DAILY_END, price_range=PRICE_RANGE, min_levels=MIN_LEVELS,
    )

    # Allocate from the existing row count, then check exact times and mid-prices.
    row_count = len(expected_times)
    clean_features = np.empty((row_count, 40), dtype=np.float64)
    perturbed_features = {}
    for probability in generators:
        perturbed_features[probability] = np.empty_like(clean_features)
    times = np.empty(row_count, dtype=np.int64)
    mid_prices = np.empty(row_count, dtype=np.float64)
    count = 0
    for row, orderbook in enumerate(orderbooks):
        times[row] = orderbook["time"].value
        mid_prices[row] = orderbook["mid_price"]
        clean_features[row] = orderbook_to_ofn(orderbook, COSTS)
        for probability, rng in generators.items():
            perturbed = perturb_orderbook(
                orderbook, probability, rng, tick_size=TICK_SIZE, n_levels=MIN_LEVELS,
            )
            perturbed_features[probability][row] = orderbook_to_ofn(perturbed, COSTS)
        count += 1
    if count != row_count:
        raise ValueError("Reconstructed row count differs from the processed day.")
    np.testing.assert_array_equal(times, expected_times)
    np.testing.assert_array_equal(mid_prices, expected_mid_prices)
    return clean_features, perturbed_features

## 4. Normalize and evaluate the full test period

Normalize every version using the same **unperturbed preceding five trading days**, with float64 statistics and float32 model inputs. For each test day, require clean reconstruction and normalization to reproduce the saved OFN features exactly. Use raw clean OFN statistics, not the price-level statistics in the processed files.

Disable gradients and dropout. Evaluate one day and probability at a time, accumulate confusion counts and sample-weighted loss, then check that class supports match the clean test. Only one day's feature arrays and five days of small statistics are retained; no perturbed-feature cache is written. The next run cell evaluates the complete stock.

In [4]:
def evaluate_stock():
    """Evaluate this stock's full test period with fixed weights and labels."""
    daily_statistics = []
    generators = {}
    confusion = {}
    loss_sum = {}
    for probability in PERTURBATION_PROBABILITIES:
        generators[probability] = np.random.default_rng(PERTURBATION_SEED)
        confusion[probability] = np.zeros((3, 3), dtype=np.int64)
        loss_sum[probability] = 0.0

    started = perf_counter()
    with pd.HDFStore(SOURCE_FILE, mode="r") as store:
        for index in range(first_day, selected_days[-1] + 1):
            date = all_dates[index]
            is_test_day = index >= selected_days[0]
            with np.load(PROCESSED_DIR / f"{date}.npz", allow_pickle=False) as arrays:
                expected_times = arrays["time_ns"]
                expected_mid_prices = arrays["mid_price"]
            # Historical days do not consume any perturbation random draws.
            raw_features, perturbed_features = prepare_ofn_day(
                store[day_keys[index]], expected_times, expected_mid_prices,
                generators if is_test_day else {},
            )
            current_statistics = (
                len(raw_features),
                raw_features.mean(axis=0, dtype=np.float64),
                raw_features.std(axis=0, ddof=0, dtype=np.float64),
            )
            if is_test_day:
                mean, std = compute_history_statistics(daily_statistics)
                with np.load(FEATURE_DIR / f"{date}.npz", allow_pickle=False) as arrays:
                    np.testing.assert_array_equal(
                        normalize_features(raw_features, mean, std).astype(np.float32),
                        arrays["features"],
                    )
                with np.load(LABEL_DIR / f"{date}.npz", allow_pickle=False) as arrays:
                    labels = arrays["labels"]
                for probability in PERTURBATION_PROBABILITIES:
                    features = normalize_features(
                        perturbed_features.pop(probability), mean, std,
                    ).astype(np.float32)
                    dataset = DailyWindowDataset(
                        [features], [labels],
                        window_length=config["window_length"], horizon=config["horizon"],
                    )
                    loader = torch.utils.data.DataLoader(
                        dataset, batch_size=config["batch_size"],
                        shuffle=False, num_workers=0, drop_last=False,
                    )
                    metrics = run_epoch(model, loader, criterion, DEVICE, log_every=0)
                    confusion[probability] += metrics["confusion"]
                    loss_sum[probability] += metrics["loss"] * len(dataset)
                    del features, dataset, loader
                del labels
                days_finished = index - selected_days[0] + 1
                print(f"{STOCK}: finished {date}, {days_finished}/{len(test_dates)} test days; {(perf_counter() - started) / 60:.1f} min", flush=True)
            else:
                print(f"{STOCK}: clean history {date} prepared.", flush=True)

            # Only the unperturbed day enters the next day's normalization history.
            daily_statistics.append(current_statistics)
            if len(daily_statistics) > HISTORY_DAYS:
                daily_statistics.pop(0)
            del raw_features, perturbed_features

    results = {0.0: baseline_metrics}
    for probability in PERTURBATION_PROBABILITIES:
        metrics = classification_metrics(confusion[probability])
        np.testing.assert_array_equal(metrics["support"], baseline_metrics["support"])
        metrics["loss"] = loss_sum[probability] / int(metrics["support"].sum())
        results[probability] = metrics
    return results

In [5]:
print(f"Starting {STOCK}: five probabilities, {len(test_dates)} test days.", flush=True)
results = evaluate_stock()
print(f"Finished {STOCK}.", flush=True)

Starting PKNORLEN: five probabilities, 62 test days.
PKNORLEN: clean history 20170925 prepared.
PKNORLEN: clean history 20170926 prepared.
PKNORLEN: clean history 20170927 prepared.
PKNORLEN: clean history 20170928 prepared.
PKNORLEN: clean history 20170929 prepared.
PKNORLEN: finished 20171002, 1/62 test days; 2.1 min
PKNORLEN: finished 20171003, 2/62 test days; 3.6 min
PKNORLEN: finished 20171004, 3/62 test days; 4.8 min
PKNORLEN: finished 20171005, 4/62 test days; 6.3 min
PKNORLEN: finished 20171006, 5/62 test days; 8.7 min
PKNORLEN: finished 20171009, 6/62 test days; 10.7 min
PKNORLEN: finished 20171010, 7/62 test days; 12.4 min
PKNORLEN: finished 20171011, 8/62 test days; 15.2 min
PKNORLEN: finished 20171012, 9/62 test days; 17.8 min
PKNORLEN: finished 20171013, 10/62 test days; 19.6 min
PKNORLEN: finished 20171016, 11/62 test days; 22.1 min
PKNORLEN: finished 20171017, 12/62 test days; 24.4 min
PKNORLEN: finished 20171018, 13/62 test days; 26.7 min
PKNORLEN: finished 20171019, 14

## 5. Compare overall scores

Use the same columns and definitions as experiment 4. Precision, recall, and F1 are weighted by true class support; weighted recall equals accuracy. Macro F1 gives equal weight to the three classes. Scores are percentages, changes from this OFN model's P=0% baseline are percentage points, and loss is mean cross-entropy.

The P=0% row reuses the saved CUDA evaluation; new rows use the device above, so small device-related differences are possible. For comparisons with experiment 4, compare both absolute scores and changes from each model's own clean baseline. This is one model and one perturbation seed per stock.

In [6]:
overall_rows = []
class_rows = []
for probability, metrics in results.items():
    support = metrics["support"]
    overall_rows.append({
        "Stock": STOCK,
        "P (%)": int(round(100 * probability)),
        "Samples": int(support.sum()),
        "Accuracy (%)": 100 * float(metrics["accuracy"]),
        "Precision weighted (%)": 100 * float(np.dot(metrics["precision"], support) / support.sum()),
        "Recall weighted (%)": 100 * float(np.dot(metrics["recall"], support) / support.sum()),
        "F1 weighted (%)": 100 * float(metrics["weighted_f1"]),
        "F1 macro (%)": 100 * float(np.mean(metrics["f1"])),
        "Loss": float(metrics["loss"]),
        "Accuracy change (pp)": 100 * float(metrics["accuracy"] - baseline_metrics["accuracy"]),
        "F1 change (pp)": 100 * float(metrics["weighted_f1"] - baseline_metrics["weighted_f1"]),
    })
    for class_index, class_name in enumerate(config["class_names"]):
        class_rows.append({
            "Stock": STOCK,
            "P (%)": int(round(100 * probability)),
            "Class": class_name,
            "Support": int(support[class_index]),
            "Precision (%)": 100 * float(metrics["precision"][class_index]),
            "Recall (%)": 100 * float(metrics["recall"][class_index]),
            "F1 (%)": 100 * float(metrics["f1"][class_index]),
        })
overall_table = pd.DataFrame(overall_rows)
class_table = pd.DataFrame(class_rows)
score_format = {column: "{:.4f}" for column in overall_table if column not in ("Stock", "P (%)", "Samples")}
display(overall_table.style.hide(axis="index").format(score_format))

Stock,P (%),Samples,Accuracy (%),Precision weighted (%),Recall weighted (%),F1 weighted (%),F1 macro (%),Loss,Accuracy change (pp),F1 change (pp)
PKNORLEN,0,4313703,66.0433,67.2724,66.0433,66.3838,65.3514,0.7704,0.0000,0.0000
PKNORLEN,1,4313703,66.0423,67.2740,66.0423,66.3832,65.3513,0.7704,-0.0010,-0.0006
PKNORLEN,5,4313703,66.0446,67.2814,66.0446,66.3863,65.3553,0.7705,0.0013,0.0025
PKNORLEN,10,4313703,66.0418,67.2857,66.0418,66.3847,65.3547,0.7705,-0.0015,0.0009
PKNORLEN,25,4313703,66.0365,67.2953,66.0365,66.3820,65.3538,0.7706,-0.0068,-0.0018
PKNORLEN,50,4313703,66.0349,67.3079,66.0349,66.3827,65.3565,0.7707,-0.0084,-0.0010


## 6. Inspect each class

Display precision, recall, F1, and support for down, stationary, and up. Labels and eligible windows remain fixed at every probability.

In [7]:
display(class_table.style.hide(axis="index").format({
    "Precision (%)": "{:.4f}", "Recall (%)": "{:.4f}", "F1 (%)": "{:.4f}",
}))

Stock,P (%),Class,Support,Precision (%),Recall (%),F1 (%)
PKNORLEN,0,down,1636304,72.6072,65.6652,68.9619
PKNORLEN,0,stationary,1039891,52.0485,65.2154,57.8928
PKNORLEN,0,up,1637508,71.6093,66.9469,69.1997
PKNORLEN,1,down,1636304,72.6116,65.6585,68.9602
PKNORLEN,1,stationary,1039891,52.0435,65.2310,57.8958
PKNORLEN,1,up,1637508,71.6123,66.9410,69.1979
PKNORLEN,5,down,1636304,72.6180,65.6528,68.9600
PKNORLEN,5,stationary,1039891,52.0371,65.2680,57.9064
PKNORLEN,5,up,1637508,71.6295,66.9291,69.1996
PKNORLEN,10,down,1636304,72.6290,65.6344,68.9548


## 7. Save this stock's results

Save `metrics.csv`, `class_metrics.csv`, and `test_metrics.npz` in `PKNORLEN/perturbation/`. These files are separate from the stock's clean training and test outputs. Rerunning this save cell replaces only these perturbation result files.

Keep experiment 4's NumPy layout: a stock axis of length one, followed by probabilities `[0, 0.01, 0.05, 0.10, 0.25, 0.50]`. Confusion rows are true classes and columns are predicted classes, ordered down, stationary, up.

In [8]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
overall_table.to_csv(OUTPUT_DIR / "metrics.csv", index=False)
class_table.to_csv(OUTPUT_DIR / "class_metrics.csv", index=False)
all_probabilities = (0.0,) + PERTURBATION_PROBABILITIES
confusion_matrices = np.stack([results[p]["confusion"] for p in all_probabilities])[None, ...]
losses = np.asarray([[results[p]["loss"] for p in all_probabilities]])
np.savez_compressed(
    OUTPUT_DIR / "test_metrics.npz",
    stocks=np.asarray([STOCK]), probabilities=np.asarray(all_probabilities),
    confusion=confusion_matrices, loss=losses,
    class_names=np.asarray(config["class_names"]),
    perturbation_seed=PERTURBATION_SEED, tick_size=TICK_SIZE, added_volume=1,
    perturbation_levels=MIN_LEVELS, history_days=HISTORY_DAYS,
    window_length=config["window_length"], horizon=config["horizon"],
    batch_size=config["batch_size"], label_alpha=np.asarray([config["label_alpha"]]),
    selected_epoch=np.asarray([checkpoint["epoch"]]), test_dates=np.asarray([test_dates]),
    feature_type="ofn", max_cost=MAX_COST, price_range=PRICE_RANGE,
    daily_start=DAILY_START.isoformat(), daily_end=DAILY_END.isoformat(),
    evaluation_device=str(DEVICE), baseline_source="experiment_05 saved CUDA evaluation",
    python=sys.version.split()[0], numpy=np.__version__, pytorch=torch.__version__,
)
print(f"Saved metrics.csv, class_metrics.csv, and test_metrics.npz in {OUTPUT_DIR.relative_to(PROJECT_DIR)}.")

Saved metrics.csv, class_metrics.csv, and test_metrics.npz in experiments/experiment_05/PKNORLEN/perturbation.
